In [ ]:
# MATLAB section 1/1 for FitResultReference: FitResult Reference

# % FitResult Reference
# The `FitResult` class stores model fitting outputs generated by
# `Analysis.RunAnalysisForNeuron` and `Analysis.RunAnalysisForAllNeurons`.
#
# This reference page is generated from the canonical runtime class:
#
# * <../FitResult.m FitResult.m>
#
# Related pages:
#
# * <FitResultExamples.html FitResult Examples>
# * <Analysis.html Analysis Reference>
# * <PaperOverview.html Paper-Aligned Toolbox Map>

# Python translation bootstrap + execution for single-section helpfile.

# Topic: FitResultReference
# Execution group: full
# Workflow family: analysis
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/FitResultReference.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "FitResultReference"
FAMILY = "analysis"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"FitResultReference: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"FitResultReference: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"FitResultReference: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"FitResultReference: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 0

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)

# FitResultReference: serialize/restore fit metadata and inspect fields.
from nstat.compat.matlab import Analysis, FitResult

dt = 0.02
t = np.arange(0.0, 12.0, dt)
x = np.column_stack([np.sin(2.0 * np.pi * 0.35 * t), np.cos(2.0 * np.pi * 0.15 * t)])
y = rng.poisson(np.exp(-2.0 + 0.9 * x[:, 0] - 0.4 * x[:, 1]) * dt)

fit_native = Analysis.fitGLM(X=x, y=y, fitType="poisson", dt=dt)
fit_native.parameter_labels = ["stim_sin", "stim_cos"]
payload = fit_native.to_structure()
fit = FitResult.fromStructure(payload)

lam_hat = fit.evalLambda(x)
coef = fit.getCoeffs()
param = fit.getParam("intercept")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].bar(np.arange(coef.size), coef, color="tab:blue")
axes[0].set_xticks(np.arange(coef.size), labels=fit.parameter_labels or ["c1", "c2"], rotation=35, ha="right")
axes[0].set_title(f"{TOPIC}: coefficients")
axes[0].set_ylabel("weight")

axes[1].plot(t, lam_hat, color="tab:green", linewidth=1.1)
axes[1].set_title("evalLambda output")
axes[1].set_xlabel("time [s]")
axes[1].set_ylabel("Hz")
plt.tight_layout()
plt.show()

assert np.isfinite(float(param))
assert lam_hat.size == t.size

CHECKPOINT_METRICS = {
    "coef_norm": float(np.linalg.norm(coef)),
    "intercept": float(param),
}
CHECKPOINT_LIMITS = {
    "coef_norm": (0.0, 100.0),
    "intercept": (-20.0, 20.0),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
